### The purpose of this notebook is to use BeautifulSoup to create files for Pokemon DB

In [1]:
%%time
import numpy as np
import pandas as pd
import bs4
import requests as rq
import re

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

CPU times: user 2.44 s, sys: 398 ms, total: 2.84 s
Wall time: 391 ms


### The base website we'll scrape from

In [2]:
%%time
url_pokedex = 'https://pokemondb.net/pokedex/national'
url_base = 'https://pokemondb.net/pokedex/'
raw_pokedex = rq.get(url_pokedex)
bs_pokedex = bs4.BeautifulSoup(raw_pokedex.text,'html.parser')

CPU times: user 242 ms, sys: 7.86 ms, total: 250 ms
Wall time: 369 ms


### Print some of the scraped site so we can see the html format

### Parse Pokemon Names

In [3]:
pokemon_names = bs_pokedex.find_all('a', class_='ent-name')
len(pokemon_names)

1025

### Name and number will be the first columns in the Pandas data frame

In [4]:
pokemon_df = pd.DataFrame()
numbers = [0]*len(pokemon_names)
names = ['']*len(pokemon_names)
for n in range(len(pokemon_names)):
    numbers[n]=n+1
    names[n] = pokemon_names[n].get_text()
pokemon_df['Number'] = numbers
pokemon_df['Name'] = names
pokemon_df.shape



(1025, 2)

In [5]:
pokemon_df.tail(10)

,Number,Name
1015,1016,Fezandipiti
1016,1017,Ogerpon
1017,1018,Archaludon
1018,1019,Hydrapple
1019,1020,Gouging Fire
1020,1021,Raging Bolt
1021,1022,Iron Boulder
1022,1023,Iron Crown
1023,1024,Terapagos
1024,1025,Pecharunt


## Fast PoC: Gen 1 matchup explorer (network-safe)


In [6]:

url_pokedex = 'https://pokemondb.net/pokedex/national'

headers = {
    'User-Agent': 'Mozilla/5.0 (compatible; PokemonScraper/1.0)'
}

# Use existing bs_pokedex if present; otherwise fetch with timeout
if 'bs_pokedex' not in globals():
    try:
        raw_pokedex = rq.get(url_pokedex, headers=headers, timeout=20)
        raw_pokedex.raise_for_status()
        bs_pokedex = bs4.BeautifulSoup(raw_pokedex.text, 'html.parser')
    except Exception as e:
        print('Network fetch failed:', e)
        bs_pokedex = None



In [7]:
# Build a small Gen 1 dataframe for a fast PoC

if bs_pokedex is None:
    # Fallback to a tiny static list so the UI still works
    gen1_df = pd.DataFrame([
        {'Number': 1, 'Name': 'Bulbasaur', 'Type1': 'Grass', 'Type2': 'Poison'},
        {'Number': 4, 'Name': 'Charmander', 'Type1': 'Fire', 'Type2': None},
        {'Number': 7, 'Name': 'Squirtle', 'Type1': 'Water', 'Type2': None},
        {'Number': 25, 'Name': 'Pikachu', 'Type1': 'Electric', 'Type2': None},
        {'Number': 39, 'Name': 'Jigglypuff', 'Type1': 'Normal', 'Type2': 'Fairy'},
        {'Number': 52, 'Name': 'Meowth', 'Type1': 'Normal', 'Type2': None},
        {'Number': 63, 'Name': 'Abra', 'Type1': 'Psychic', 'Type2': None},
        {'Number': 92, 'Name': 'Gastly', 'Type1': 'Ghost', 'Type2': 'Poison'},
        {'Number': 133, 'Name': 'Eevee', 'Type1': 'Normal', 'Type2': None},
        {'Number': 150, 'Name': 'Mewtwo', 'Type1': 'Psychic', 'Type2': None},
    ])
else:
    infocards = bs_pokedex.select('div.infocard')
    rows = []
    for card in infocards:
        num_tag = card.select_one('small')
        name_tag = card.select_one('a.ent-name')
        type_tags = card.select('a[href^="/type/"]')
        if not (num_tag and name_tag and type_tags):
            continue
        try:
            number = int(num_tag.get_text(strip=True).lstrip('#'))
        except ValueError:
            continue
        types = [t.get_text(strip=True) for t in type_tags]
        type_1 = types[0] if len(types) > 0 else None
        type_2 = types[1] if len(types) > 1 else None
        rows.append({
            'Number': number,
            'Name': name_tag.get_text(strip=True),
            'Type1': type_1,
            'Type2': type_2,
        })

    pokemon_types_df = pd.DataFrame(rows).sort_values('Number')
    gen1_df = pokemon_types_df[pokemon_types_df['Number'] <= 151].reset_index(drop=True)

# Keep the UI fast by limiting to first 25 Gen 1 mons
if len(gen1_df) > 25:
    gen1_df = gen1_df.iloc[:25].reset_index(drop=True)

print('Loaded rows:', len(gen1_df))


Loaded rows: 25


In [8]:
# Type chart (modern multipliers)
TYPE_CHART = {
    'Normal':  {'Rock': 0.5, 'Ghost': 0.0, 'Steel': 0.5},
    'Fire':    {'Fire': 0.5, 'Water': 0.5, 'Grass': 2.0, 'Ice': 2.0, 'Bug': 2.0, 'Rock': 0.5, 'Dragon': 0.5, 'Steel': 2.0},
    'Water':   {'Fire': 2.0, 'Water': 0.5, 'Grass': 0.5, 'Ground': 2.0, 'Rock': 2.0, 'Dragon': 0.5},
    'Electric':{'Water': 2.0, 'Electric': 0.5, 'Grass': 0.5, 'Ground': 0.0, 'Flying': 2.0, 'Dragon': 0.5},
    'Grass':   {'Fire': 0.5, 'Water': 2.0, 'Grass': 0.5, 'Poison': 0.5, 'Ground': 2.0, 'Flying': 0.5, 'Bug': 0.5, 'Rock': 2.0, 'Dragon': 0.5, 'Steel': 0.5},
    'Ice':     {'Fire': 0.5, 'Water': 0.5, 'Grass': 2.0, 'Ground': 2.0, 'Flying': 2.0, 'Dragon': 2.0, 'Steel': 0.5},
    'Fighting':{'Normal': 2.0, 'Ice': 2.0, 'Rock': 2.0, 'Dark': 2.0, 'Steel': 2.0, 'Poison': 0.5, 'Flying': 0.5, 'Psychic': 0.5, 'Bug': 0.5, 'Ghost': 0.0, 'Fairy': 0.5},
    'Poison':  {'Grass': 2.0, 'Fairy': 2.0, 'Poison': 0.5, 'Ground': 0.5, 'Rock': 0.5, 'Ghost': 0.5, 'Steel': 0.0},
    'Ground':  {'Fire': 2.0, 'Electric': 2.0, 'Grass': 0.5, 'Poison': 2.0, 'Flying': 0.0, 'Bug': 0.5, 'Rock': 2.0, 'Steel': 2.0},
    'Flying':  {'Electric': 0.5, 'Grass': 2.0, 'Fighting': 2.0, 'Bug': 2.0, 'Rock': 0.5, 'Steel': 0.5},
    'Psychic': {'Fighting': 2.0, 'Poison': 2.0, 'Psychic': 0.5, 'Steel': 0.5, 'Dark': 0.0},
    'Bug':     {'Grass': 2.0, 'Psychic': 2.0, 'Dark': 2.0, 'Fire': 0.5, 'Fighting': 0.5, 'Poison': 0.5, 'Flying': 0.5, 'Ghost': 0.5, 'Steel': 0.5, 'Fairy': 0.5},
    'Rock':    {'Fire': 2.0, 'Ice': 2.0, 'Flying': 2.0, 'Bug': 2.0, 'Fighting': 0.5, 'Ground': 0.5, 'Steel': 0.5},
    'Ghost':   {'Psychic': 2.0, 'Ghost': 2.0, 'Dark': 0.5, 'Normal': 0.0},
    'Dragon':  {'Dragon': 2.0, 'Steel': 0.5, 'Fairy': 0.0},
    'Dark':    {'Psychic': 2.0, 'Ghost': 2.0, 'Fighting': 0.5, 'Dark': 0.5, 'Fairy': 0.5},
    'Steel':   {'Ice': 2.0, 'Rock': 2.0, 'Fairy': 2.0, 'Fire': 0.5, 'Water': 0.5, 'Electric': 0.5, 'Steel': 0.5},
    'Fairy':   {'Fighting': 2.0, 'Dragon': 2.0, 'Dark': 2.0, 'Fire': 0.5, 'Poison': 0.5, 'Steel': 0.5},
}

ATTACK_TYPES = list(TYPE_CHART.keys())


In [ ]:

# Defensive multipliers for a given Pokemon's types
def defensive_multipliers(def_types):
    multipliers = {atk: 1.0 for atk in ATTACK_TYPES}
    for atk in ATTACK_TYPES:
        for d in def_types:
            if d is None:
                continue
            multipliers[atk] *= TYPE_CHART.get(atk, {}).get(d, 1.0)
    return multipliers

# Color bars by multiplier
def multiplier_color(value):
    if value == 0:
        return '#4B5563'  # immune
    if value >= 4:
        return '#B91C1C'  # 4x
    if value >= 2:
        return '#DC2626'  # 2x
    if value <= 0.25:
        return '#059669'  # 0.25x
    if value <= 0.5:
        return '#10B981'  # 0.5x
    return '#9CA3AF'     # neutral


def plot_matchups(pokemon_name):
    row = gen1_df[gen1_df['Name'] == pokemon_name].iloc[0]
    types = [row['Type1'], row['Type2']]
    multipliers = defensive_multipliers(types)
    df = pd.DataFrame({
        'Attack Type': list(multipliers.keys()),
        'Multiplier': list(multipliers.values()),
    })

    colors = [multiplier_color(v) for v in df['Multiplier']]

    plt.figure(figsize=(12, 4))
    plt.bar(df['Attack Type'], df['Multiplier'], color=colors)
    plt.axhline(1.0, color='#111827', linewidth=1)
    title_types = ' / '.join([t for t in types if t])
    plt.title(f"{pokemon_name} ({title_types}) defensive matchups")
    plt.ylabel('Damage multiplier')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, max(4, df['Multiplier'].max() + 0.5))
    plt.show()

# Widget UI
dropdown = widgets.Dropdown(
    options=gen1_df['Name'].tolist(),
    value=gen1_df['Name'].iloc[0],
    description='Pokemon',
    layout=widgets.Layout(width='300px')
)
output = widgets.Output()


def on_change(change):
    if change['name'] == 'value':
        with output:
            output.clear_output()
            plot_matchups(change['new'])


dropdown.observe(on_change, names='value')

with output:
    plot_matchups(dropdown.value)

display(dropdown, output)


Dropdown(description='Pokemon', layout=Layout(width='300px'), options=('Bulbasaur', 'Ivysaur', 'Venusaur', 'Ch…

Output()